# SylCipher reproduction on small number of utterances

This notebook extracts a tiny aligned subset from $PUSM_SOURCE_DIR, trains the discrete SylCipher without differentiable boundary detector with plain PyTorch, and reports UER plus training statistics after every epoch. It has **no Fairseq, Hydra, Slurm, or interleave/continuous-feature dependency**.

The training signal is unpaired: speech cluster sequences and text sequences are shuffled independently. References are used only to compute the diagnostic subset UER. With a small number of training utterances, the result is a pipeline check rather than a meaningful ASR result.

## Colab/local setup

On the SLS workspace the default path below works directly. In Colab, mount Drive or upload/copy the `top_${clus_size}` directory, then set `PUSM_SOURCE_DIR` before running the configuration cell. PyTorch, NumPy, pandas, and matplotlib are already included in standard Colab runtimes.

In [ ]:
# Optional in Google Colab:
from google.colab import drive
import zipfile
!pip -q install -U gdown
!gdown https://drive.google.com/file/d/1S5sw4WsnDu9XMj-vib_SicsSQpyBdRP-/view?usp=sharing -O archive.zip
#!gdown https://drive.google.com/file/d/1pVkt0LVgEjxDYvtb-y4V1cg_tnDRMMvI/view?usp=sharing -O archive.zip  # LibriSpeech subset 10
#!gdown https://drive.google.com/file/d/1fqQSO56OB5u5MkgShrQ_sE9z60lRCc8z/view?usp=sharing -O archive.zip  # LibriSpeech subset 1000
!unzip -q archive.zip -d ./

#PUSM_WORK_DIR="/content/colab_suisiann"
#PUSM_WORK_DIR="/content/colab_libri_subset_10"
PUSM_WORK_DIR="/content/colab_libri_subset_1000"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00
Downloading...
From: https://drive.google.com/uc?id=1S5sw4WsnDu9XMj-vib_SicsSQpyBdRP-
To: /content/archive.zip
100% 171k/171k [00:00<00:00, 17.8MB/s]


In [ ]:
import json, math, os, random, time, warnings
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class Config:
    source_dir: str = os.environ.get(
        'PUSM_SOURCE_DIR',
        '/data/sls/scratch/limingw/workplace/VGUnsupASR/manifest/'
        'suisiann_vad/feat_xeus_L18_sylber_ft_bnd/top_1024',
    #    'librispeech_clean_vad/feat_hubert_large_L21_sylber_bnd/top_2048',
    )
    work_dir: str = PUSM_WORK_DIR
    num_utterances: int = 2775
    num_speech_clusters: int = 1024
    fallback_active_clusters: int = 32
    fallback_kmeans_iterations: int = 8
    seed: int = 2
    epochs: int = int(os.environ.get('PUSM_EPOCHS', '200'))
    # None = every microbatch contains the entire subset.
    batch_size: int | None = (
        None if os.environ.get('PUSM_BATCH_SIZE', '5').lower() in {'none', 'all', 'full'}
        else int(os.environ.get('PUSM_BATCH_SIZE', '5'))
    )
    # Accumulate this many complete corpus gradient passes before optimizer.step().
    # One global gradient pass per epoch; the last partial group is flushed.
    update_frequency: int = int(os.environ.get('PUSM_UPDATE_FREQUENCY', '2'))
    learning_rate: float = 0.4
    skipgram_size: int = 4
    max_text_positions: int = 100
    adam_betas: tuple = (0.5, 0.98)
    adam_eps: float = 1e-6

cfg = Config()
assert cfg.update_frequency >= 1
assert cfg.batch_size is None or cfg.batch_size >= 1
SOURCE_DIR, WORK_DIR = Path(cfg.source_dir), Path(cfg.work_dir)
WORK_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(json.dumps({**asdict(cfg), 'device': str(device)}, indent=2))

{
  "source_dir": "/data/sls/scratch/limingw/workplace/VGUnsupASR/manifest/suisiann_vad/feat_xeus_L18_sylber_ft_bnd/top_1024",
  "work_dir": "/content/colab_suisiann",
  "num_utterances": 2775,
  "num_speech_clusters": 1024,
  "fallback_active_clusters": 32,
  "fallback_kmeans_iterations": 8,
  "seed": 2,
  "epochs": 2000,
  "batch_size": 5,
  "update_frequency": 2,
  "learning_rate": 0.4,
  "skipgram_size": 4,
  "max_text_positions": 100,
  "adam_betas": [
    0.5,
    0.98
  ],
  "adam_eps": 1e-06,
  "device": "cuda"
}


## Prepare a portable small subset

The compact on-disk format is deliberately simple: concatenated `train.codes.npy`, one length per line in `train.lengths`, aligned `train.files` and `train.text`, plus `dict.txt`, `vocab.json`, and `manifest.json`.

The current workspace copy has broken links for the \${clus_size}-way `.km` labels and codebook. The preparation code therefore prefers the real `.km` file when present, but has an explicit NumPy k-means fallback using only the selected continuous segment features. The fallback retains a \${clus_size}-row input embedding while activating only `fallback_active_clusters` rows; `manifest.json` records which path was used.

In [ ]:
SPECIALS = ['<s>', '<pad>', '</s>', '<unk>']
NSPECIAL = len(SPECIALS)

def read_first_lines(path, n):
    with Path(path).open(encoding='utf-8') as f:
        return [next(f).rstrip('\n') for _ in range(n)]

def read_lengths(path, n):
    return np.asarray([int(x) for x in read_first_lines(path, n)], dtype=np.int64)

def tiny_kmeans(x, n_clusters, iterations, seed):
    # Dependency-free fallback for this tiny demo; not a replacement for the full codebook.
    rng = np.random.default_rng(seed)
    k = min(n_clusters, len(x))
    centers = x[rng.choice(len(x), size=k, replace=False)].astype(np.float32).copy()
    labels = np.zeros(len(x), dtype=np.int64)
    x2 = (x * x).sum(1, keepdims=True)
    for _ in range(iterations):
        c2 = (centers * centers).sum(1)[None, :]
        labels = (x2 + c2 - 2.0 * x @ centers.T).argmin(1).astype(np.int64)
        counts = np.bincount(labels, minlength=k)
        sums = np.zeros_like(centers)
        np.add.at(sums, labels, x)
        nonempty = counts > 0
        centers[nonempty] = sums[nonempty] / counts[nonempty, None]
        centers[~nonempty] = x[rng.integers(0, len(x), size=(~nonempty).sum())]
    return labels, centers

def prepare_subset(cfg, source, out):
    km_dir = source / f'km_dir_{cfg.num_speech_clusters}'
    seg_dir = km_dir / 'seg_feats'
    lengths = read_lengths(seg_dir / 'train_0_1.len', cfg.num_utterances)
    files = read_first_lines(
        km_dir / 'discrete_speech_train_clsattnbndkmeans/file_list.txt',
        cfg.num_utterances,
    )
    texts = read_first_lines(source / 'train_words.txt', cfg.num_utterances)
    total = int(lengths.sum())

    km_candidates = [
        seg_dir / 'km_idx_clsattnbndkmeans/train.km',
        km_dir / 'discrete_speech_train_clsattnbndkmeans/data.km',
    ]
    km_path = next((p for p in km_candidates if p.exists()), None)
    centers = None
    if km_path is not None:
        code_lines = read_first_lines(km_path, cfg.num_utterances)
        code_seqs = [np.fromstring(line, sep=' ', dtype=np.int64) for line in code_lines]
        got = np.asarray([len(x) for x in code_seqs])
        if not np.array_equal(got, lengths):
            raise ValueError(f'.km/feature length mismatch: {got.tolist()} != {lengths.tolist()}')
        codes = np.concatenate(code_seqs)
        quantization = f'existing labels: {km_path}'
    else:
        feature_path = seg_dir / 'train_0_1.npy'
        all_features = np.load(feature_path, mmap_mode='r')
        features = np.asarray(all_features[:total], dtype=np.float32).copy()
        codes, centers = tiny_kmeans(
            features, cfg.fallback_active_clusters, cfg.fallback_kmeans_iterations, cfg.seed
        )
        quantization = f'fallback tiny k-means ({len(centers)} active / {cfg.num_speech_clusters} slots)'
        np.save(out / 'fallback_centers.npy', centers)
        warnings.warn('The requested labels/codebook are missing; used the documented tiny fallback.')

    if codes.min() < 0 or codes.max() >= cfg.num_speech_clusters:
        raise ValueError(f'cluster IDs must lie in [0, {cfg.num_speech_clusters})')

    source_symbols = [line.rsplit(maxsplit=1)[0] for line in (source / 'dict.txt').read_text().splitlines()]
    observed = set(' '.join(texts).split())
    symbols = SPECIALS + [s for s in source_symbols if s in observed and s not in SPECIALS]
    symbols += sorted(observed.difference(symbols))
    token_to_id = {token: i for i, token in enumerate(symbols)}
    text_ids = [[token_to_id.get(t, token_to_id['<unk>']) for t in line.split()] for line in texts]

    np.save(out / 'train.codes.npy', codes.astype(np.int64))
    (out / 'train.lengths').write_text(''.join(f'{x}\n' for x in lengths))
    (out / 'train.files').write_text(''.join(f'{x}\n' for x in files))
    (out / 'train.text').write_text(''.join(f'{x}\n' for x in texts))
    (out / 'dict.txt').write_text(''.join(f'{s} 1\n' for s in symbols[NSPECIAL:]))
    (out / 'vocab.json').write_text(json.dumps(symbols, indent=2))
    manifest = {
        'source': str(source.resolve()), 'utterances': len(lengths),
        'segments': total, 'vocab_size': len(symbols),
        'num_speech_cluster_slots': cfg.num_speech_clusters,
        'active_speech_clusters': int(np.unique(codes).size),
        'quantization': quantization, 'files': files,
    }
    (out / 'manifest.json').write_text(json.dumps(manifest, indent=2))
    return lengths, codes, texts, text_ids, symbols, manifest

if not SOURCE_DIR.exists():
    raise FileNotFoundError(f'Set PUSM_SOURCE_DIR to the top_${clus_size} directory; not found: {SOURCE_DIR}')
lengths, flat_codes, texts, text_ids, symbols, subset_manifest = prepare_subset(cfg, SOURCE_DIR, WORK_DIR)
print(json.dumps(subset_manifest, indent=2))

{
  "source": "/data/sls/scratch/limingw/workplace/VGUnsupASR/manifest/suisiann_vad/feat_xeus_L18_sylber_ft_bnd/top_1024",
  "utterances": 2775,
  "segments": 46964,
  "vocab_size": 1009,
  "num_speech_cluster_slots": 1024,
  "active_speech_clusters": 1024,
  "quantization": "existing labels: /data/sls/scratch/limingw/workplace/VGUnsupASR/manifest/suisiann_vad/feat_xeus_L18_sylber_ft_bnd/top_1024/km_dir_1024/seg_feats/km_idx_clsattnbndkmeans/train.km",
  "files": [
    "SuiSiann_3297.wav",
    "SuiSiann_2537.wav",
    "SuiSiann_3158.wav",
    "SuiSiann_0522.wav",
    "SuiSiann_1157.wav",
    "SuiSiann_1408.wav",
    "SuiSiann_0993.wav",
    "SuiSiann_0607.wav",
    "SuiSiann_2089.wav",
    "SuiSiann_0056.wav",
    "SuiSiann_2951.wav",
    "SuiSiann_1642.wav",
    "SuiSiann_2093.wav",
    "SuiSiann_0398.wav",
    "SuiSiann_2571.wav",
    "SuiSiann_1275.wav",
    "SuiSiann_2074.wav",
    "SuiSiann_3445.wav",
    "SuiSiann_1160.wav",
    "SuiSiann_0681.wav",
    "SuiSiann_0223.wav",
    "

In [ ]:
offsets = np.r_[0, np.cumsum(lengths)]
code_sequences = [
    torch.as_tensor(flat_codes[offsets[i]:offsets[i + 1]], dtype=torch.long, device=device)
    for i in range(len(lengths))
]
reference_sequences = [torch.tensor(x, dtype=torch.long, device=device) for x in text_ids]
preview = pd.DataFrame({
    'file': subset_manifest['files'], 'speech_segments': lengths,
    'text_units': [len(x) for x in text_ids], 'text': texts,
})
display(preview)

,file,speech_segments,text_units,text
0,SuiSiann_3297.wav,7,6,li tioh ai tsiah hoo pa
1,SuiSiann_2537.wav,7,7,kah gua tse long tse tsiah khui
2,SuiSiann_3158.wav,15,13,tai tsi tso be ho tshun i an ne suann hing sua...
3,SuiSiann_0522.wav,7,7,long tsong ka ia ia tshut lai
4,SuiSiann_1157.wav,22,22,d ke sng nng e sio ua e gi su e mi kah cr koh ...
...,...,...,...,...
2770,SuiSiann_2919.wav,10,9,au <OOV> khuann e sai than kua tsinn bo
2771,SuiSiann_0544.wav,4,4,gua khit lai ah
2772,SuiSiann_1745.wav,13,12,ah to bo tai tsi theh mih kiann beh khi hoo lang
2773,SuiSiann_0750.wav,14,13,oo ba sang li ho li beh ai e mih kiann ti tsia


## Stage 1: Global PUSM objective

Collect raw positional unigram and lagged skip-gram counts across all batches. Then backpropagate the exact global L1 gradient batch by batch. All four skip-gram L1 terms are summed. Batch size controls memory use, not the objective.


In [ ]:
class DiscretePUSM(nn.Module):
    def __init__(self, num_clusters, vocab_size):
        super().__init__()
        self.proj = nn.Embedding(num_clusters, vocab_size)
        nn.init.normal_(self.proj.weight, mean=0.0, std=0.02)

    def forward(self, cluster_ids):
        return self.proj(cluster_ids).softmax(-1)

def one_hot_text(sequence, vocab_size):
    return F.one_hot(sequence, num_classes=vocab_size).float()

def raw_counts(sequences, vocab_size, config):
    uni = sequences[0].new_zeros(config.max_text_positions, vocab_size)
    skip = sequences[0].new_zeros(config.skipgram_size, vocab_size, vocab_size)
    for sequence in sequences:
        size = min(len(sequence), config.max_text_positions)
        uni[:size] = uni[:size] + sequence[:size]
        for lag in range(1, config.skipgram_size + 1):
            if len(sequence) > lag:
                skip[lag-1] = skip[lag-1] + sequence[:-lag].T @ sequence[lag:]
    return uni, skip

@torch.no_grad()
def collect_global_counts(model, batches, config, vocab_size):
    speech_global, text_global = None, None
    for speech_idx, text_idx in batches:
        speech = raw_counts([model(code_sequences[int(i)]) for i in speech_idx], vocab_size, config)
        text = raw_counts([one_hot_text(reference_sequences[int(i)], vocab_size) for i in text_idx], vocab_size, config)
        if speech_global is None:
            speech_global = [torch.zeros_like(x) for x in speech]
            text_global = [torch.zeros_like(x) for x in text]
        for dest, value in zip(speech_global, speech): dest.add_(value)
        for dest, value in zip(text_global, text): dest.add_(value)
    return speech_global, text_global

def global_count_gradient(model, speech_idx, signs, config, vocab_size):
    # Exact global L1 gradient: sum_b sign(C_global - T) * dC_b/dtheta.
    counts = raw_counts([model(code_sequences[int(i)]) for i in speech_idx], vocab_size, config)
    return sum((count * sign).sum() for count, sign in zip(counts, signs))

torch.manual_seed(cfg.seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(cfg.seed)
model = DiscretePUSM(cfg.num_speech_clusters, len(symbols)).to(device)
print(model)
print(f'trainable parameters: {sum(p.numel() for p in model.parameters()):,}')

DiscretePUSM(
  (proj): Embedding(1024, 1009)
)
trainable parameters: 1,033,216


## Metrics

UER is ordinary Levenshtein distance over syllable/unit tokens divided by reference length, matching the original task's default non-CTC evaluation. Predictions with Fairseq-style special IDs 0-3 are ignored. `code_ppl` is the perplexity of hard predicted unit usage; `prob_ppl` is the entropy perplexity of the mean soft distribution.

In [ ]:
def edit_distance(a, b):
    previous = list(range(len(b) + 1))
    for i, ai in enumerate(a, 1):
        current = [i]
        for j, bj in enumerate(b, 1):
            current.append(min(current[-1] + 1, previous[j] + 1, previous[j - 1] + (ai != bj)))
        previous = current
    return previous[-1]

@torch.no_grad()
def evaluate(model, speech, references, nspecial=NSPECIAL):
    model.eval()
    errors = ref_units = pred_units = 0
    all_hard, all_probs, hypotheses = [], [], []
    for clusters, reference in zip(speech, references):
        probs = model(clusters)
        pred = probs.argmax(-1)
        pred = pred[pred >= nspecial]
        ref = reference[reference >= nspecial]
        p, r = pred.tolist(), ref.tolist()
        errors += edit_distance(p, r)
        ref_units += len(r)
        pred_units += len(p)
        all_hard.append(pred)
        all_probs.append(probs)
        hypotheses.append(p)
    hard = torch.cat(all_hard) if all_hard else torch.empty(0, dtype=torch.long, device=device)
    probs = torch.cat(all_probs)
    hard_hist = torch.bincount(hard, minlength=len(symbols)).float()[nspecial:]
    hard_dist = hard_hist / hard_hist.sum().clamp_min(1)
    mean_prob = probs.mean(0)
    code_ppl = torch.exp(-(hard_dist * (hard_dist + 1e-12).log()).sum())
    prob_ppl = torch.exp(-(mean_prob * (mean_prob + 1e-12).log()).sum())
    return {
        'uer': 100.0 * errors / max(ref_units, 1), 'unit_errors': errors,
        'reference_units': ref_units, 'predicted_units': pred_units,
        'active_predicted_units': int((hard_hist > 0).sum()),
        'vocab_seen_pct': 100.0 * float((hard_hist > 0).sum()) / max(len(symbols) - nspecial, 1),
        'code_ppl': float(code_ppl), 'prob_ppl': float(prob_ppl),
        'hypotheses': hypotheses,
    }

## Global gradient accumulation

Every epoch first collects global counts, then backpropagates using the global difference with fixed model weights. `update_frequency` counts complete corpus gradient passes per optimizer update. The last partial group is averaged and flushed. `batch_size=None` processes the full dataset as one batch. Logged losses are global counts before the update; UER is evaluated afterward.


In [ ]:
def batches_for_epoch(n, batch_size, rng):
    size = n if batch_size is None else min(batch_size, n)
    speech_order = rng.permutation(n)
    text_order = rng.permutation(n)  # deliberately unpaired
    return [(speech_order[i:i + size], text_order[i:i + size]) for i in range(0, n, size)]

def gradient_norm(parameters):
    values = [p.grad.detach().norm(2) ** 2 for p in parameters if p.grad is not None]
    return float(torch.stack(values).sum().sqrt()) if values else 0.0

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(cfg.seed)
rng = np.random.default_rng(cfg.seed)
optimizer = torch.optim.Adam(
    model.parameters(), lr=cfg.learning_rate, betas=cfg.adam_betas, eps=cfg.adam_eps
)
optimizer.zero_grad(set_to_none=True)
history, num_updates, pending_global_passes = [], 0, 0
best_uer = math.inf

for epoch in range(1, cfg.epochs + 1):
    started = time.time()
    model.train()
    losses, uni_losses, skip_losses, grad_norms = [], [], [], []
    updates_at_start = num_updates
    epoch_batches = batches_for_epoch(len(code_sequences), cfg.batch_size, rng)
    # Count pass and gradient pass use the same fixed model weights.
    speech_global, text_global = collect_global_counts(model, epoch_batches, cfg, len(symbols))
    differences = [speech - text for speech, text in zip(speech_global, text_global)]
    signs = [difference.sign() for difference in differences]
    unigram_l1, skipgram_l1 = [float(difference.abs().sum()) for difference in differences]
    global_l1 = unigram_l1 + skipgram_l1
    for speech_idx, _ in epoch_batches:
        surrogate = global_count_gradient(model, speech_idx, signs, cfg, len(symbols))
        (surrogate / cfg.update_frequency).backward()
    pending_global_passes += 1
    if pending_global_passes == cfg.update_frequency or epoch == cfg.epochs:
        if pending_global_passes < cfg.update_frequency:
            for parameter in model.parameters():
                if parameter.grad is not None:
                    parameter.grad.mul_(cfg.update_frequency / pending_global_passes)
        grad_norms.append(gradient_norm(model.parameters()))
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        pending_global_passes = 0
        num_updates += 1

    metrics = evaluate(model, code_sequences, reference_sequences)
    row = {
        'epoch': epoch, 'num_updates': num_updates,
        'updates_this_epoch': num_updates - updates_at_start,
        'pending_global_passes': pending_global_passes,
        'microbatches': len(epoch_batches),
        'effective_batch_size': len(code_sequences) * cfg.update_frequency,
        'global_utterances': len(code_sequences),
        'loss': global_l1, 'unigram_l1': unigram_l1, 'skipgram_l1': skipgram_l1,
        'grad_norm': float(np.mean(grad_norms)) if grad_norms else float('nan'),
        'lr': optimizer.param_groups[0]['lr'], 'seconds': time.time() - started,
        **{k: v for k, v in metrics.items() if k != 'hypotheses'},
    }
    history.append(row)
    print(json.dumps(row, allow_nan=True))
    example_id = 0
    example_ref = ' '.join(symbols[x] for x in reference_sequences[example_id].tolist())
    example_hyp = ' '.join(symbols[x] for x in metrics['hypotheses'][example_id])
    print(f'  example REF: {example_ref}')
    print(f'  example HYP: {example_hyp}')
    if metrics['uer'] < best_uer:
        best_uer = metrics['uer']
        torch.save({'model': model.state_dict(), 'config': asdict(cfg), 'symbols': symbols, 'epoch': epoch,
                    'num_updates': num_updates, 'uer': best_uer}, WORK_DIR / 'checkpoint_best.pt')

torch.save({'model': model.state_dict(), 'config': asdict(cfg), 'symbols': symbols,
            'epoch': cfg.epochs, 'num_updates': num_updates}, WORK_DIR / 'checkpoint_last.pt')
history_df = pd.DataFrame(history)
history_df.to_csv(WORK_DIR / 'history.csv', index=False)
display(history_df)

{"epoch": 1, "num_updates": 0, "updates_this_epoch": 0, "pending_global_passes": 1, "microbatches": 555, "effective_batch_size": 5550, "global_utterances": 2775, "loss": 206090.2578125, "unigram_l1": 22613.0390625, "skipgram_l1": 183477.21875, "grad_norm": NaN, "lr": 0.4, "seconds": 12.764079809188843, "uer": 103.37490829053559, "unit_errors": 45088, "reference_units": 43616, "predicted_units": 46964, "active_predicted_units": 56, "vocab_seen_pct": 5.5721393034825875, "code_ppl": 1.5950902700424194, "prob_ppl": 199.0697784423828}
  example REF: li tioh ai tsiah hoo pa
  example HYP: gua e e e e e penn
{"epoch": 2, "num_updates": 1, "updates_this_epoch": 1, "pending_global_passes": 0, "microbatches": 555, "effective_batch_size": 5550, "global_utterances": 2775, "loss": 206090.259765625, "unigram_l1": 22613.041015625, "skipgram_l1": 183477.21875, "grad_norm": 187.19381713867188, "lr": 0.4, "seconds": 12.973518371582031, "uer": 103.15939104915627, "unit_errors": 44994, "reference_units": 

KeyboardInterrupt: 

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
history_df.plot(x='epoch', y=['loss', 'unigram_l1', 'skipgram_l1'], ax=axes[0], title='PUSM losses')
history_df.plot(x='epoch', y='uer', ax=axes[1], title='Subset UER (%)')
history_df.plot(x='epoch', y=['code_ppl', 'prob_ppl'], ax=axes[2], title='Unit usage perplexity')
plt.tight_layout()
plt.show()

final = evaluate(model, code_sequences, reference_sequences)
for i in range(min(3, len(texts))):
    ref = ' '.join(symbols[x] for x in reference_sequences[i].tolist())
    hyp = ' '.join(symbols[x] for x in final['hypotheses'][i])
    print(f'[{i}] REF: {ref}')
    print(f'[{i}] HYP: {hyp}\n')
print('Artifacts:', WORK_DIR.resolve())
print(sorted(p.name for p in WORK_DIR.iterdir()))

## Stage 2: interleaved masked language modeling

This compact second stage mirrors the central joint-MLM path in `SpeechT5_transfer_learning`: convert speech clusters to text-unit IDs with the trained PUSM projection, mix speech-derived and paired text units positionwise at a 50/50 ratio, corrupt spans, and predict the original units with a shared Transformer encoder. It also retains the separate speech-only and text-only MLM losses; as in the reference criterion, the joint loss has weight 5.

The original implementation has separate speech/text prenets and output heads. Here both modalities already occupy the PUSM text-unit vocabulary, so a modality embedding plus one tied output head is the smallest equivalent. Reference sequences are monotonically resampled to the speech-segment length before positionwise interleaving.

In [ ]:
@dataclass
class MLMConfig:
    epochs: int = int(os.environ.get('PUSM_MLM_EPOCHS', '10'))
    batch_size: int | None = (
        None if os.environ.get('PUSM_MLM_BATCH_SIZE', 'full').lower() in {'none', 'all', 'full'}
        else int(os.environ.get('PUSM_MLM_BATCH_SIZE', 'full'))
    )
    update_frequency: int = int(os.environ.get('PUSM_MLM_UPDATE_FREQUENCY', '1'))
    learning_rate: float = 3e-4
    d_model: int = 96
    nhead: int = 4
    layers: int = 2
    feedforward_dim: int = 192
    dropout: float = 0.1
    max_positions: int = 256
    mask_prob: float = 0.30
    mean_mask_span: float = 3.5
    random_replace_prob: float = 0.10
    interleave_ratio: float = 0.50
    speech_weight: float = 1.0
    text_weight: float = 1.0
    joint_weight: float = 5.0
    seed: int = cfg.seed + 100

mlm_cfg = MLMConfig()
assert mlm_cfg.update_frequency >= 1
assert mlm_cfg.batch_size is None or mlm_cfg.batch_size >= 1
assert mlm_cfg.d_model % mlm_cfg.nhead == 0

@torch.no_grad()
def pusm_units(pusm_model, cluster_sequences):
    pusm_model.eval()
    result = []
    for clusters in cluster_sequences:
        prediction = pusm_model(clusters).argmax(-1)
        # Preserve sequence length: map predicted Fairseq specials to <unk>.
        prediction = torch.where(prediction < NSPECIAL, prediction.new_full((), SPECIALS.index('<unk>')), prediction)
        result.append(prediction)
    return result

# Freeze Stage-1 predictions, preserving utterance order and one unit per segment.
# The joint branch uses each utterance's OWN prediction, never its reference.
speech_unit_sequences = pusm_units(model, code_sequences)
pusm_predicted_text_sequences = [sequence.detach().clone() for sequence in speech_unit_sequences]
# Separate text-only MLM corpus. It is not paired with speech in the joint branch.
unpaired_text_sequences = [sequence.detach().clone() for sequence in reference_sequences]
assert len(pusm_predicted_text_sequences) == len(code_sequences)
assert all(len(text) == len(speech) for text, speech in zip(pusm_predicted_text_sequences, code_sequences))
mlm_symbols = symbols + ['<mask>']
MASK_ID, PAD_ID = len(symbols), SPECIALS.index('<pad>')
print(json.dumps(asdict(mlm_cfg), indent=2))
print(f'MLM vocabulary: {len(mlm_symbols)}; maximum sequence: {max(map(len, speech_unit_sequences))}')

In [ ]:
class InterleavedMLM(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, config.d_model, padding_idx=PAD_ID)
        self.modality_embedding = nn.Embedding(2, config.d_model)  # 0=speech, 1=text
        self.position_embedding = nn.Embedding(config.max_positions, config.d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=config.d_model, nhead=config.nhead, dim_feedforward=config.feedforward_dim,
            dropout=config.dropout, activation='gelu', batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=config.layers)
        self.layer_norm = nn.LayerNorm(config.d_model)
        self.output = nn.Linear(config.d_model, vocab_size, bias=False)
        nn.init.normal_(self.token_embedding.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.modality_embedding.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.position_embedding.weight, mean=0.0, std=0.02)
        with torch.no_grad(): self.token_embedding.weight[PAD_ID].zero_()
        self.output.weight = self.token_embedding.weight

    def forward(self, tokens, modalities, padding_mask):
        positions = torch.arange(tokens.size(1), device=tokens.device)[None, :]
        hidden = self.token_embedding(tokens) + self.modality_embedding(modalities)
        hidden = hidden + self.position_embedding(positions)
        hidden = self.encoder(hidden, src_key_padding_mask=padding_mask)
        return self.output(self.layer_norm(hidden))

def mask_positions(length, config, rng):
    target = max(1, int(round(length * config.mask_prob)))
    selected = np.zeros(length, dtype=bool)
    while selected.sum() < target:
        start = int(rng.integers(0, length))
        span = max(1, int(rng.poisson(config.mean_mask_span)))
        selected[start:min(length, start + span)] = True
    return selected

def corrupt_sequence(sequence, config, rng):
    selected_np = mask_positions(len(sequence), config, rng)
    selected = torch.as_tensor(selected_np, device=sequence.device)
    corrupted = sequence.clone()
    corrupted[selected] = MASK_ID
    selected_indices = np.flatnonzero(selected_np)
    random_flags = rng.random(len(selected_indices)) < config.random_replace_prob
    random_positions = selected_indices[random_flags]
    if len(random_positions):
        replacements = rng.integers(NSPECIAL, MASK_ID, size=len(random_positions))
        corrupted[torch.as_tensor(random_positions, device=sequence.device)] = torch.as_tensor(
            replacements, dtype=torch.long, device=sequence.device
        )
    targets = sequence.new_full(sequence.shape, -100)
    targets[selected] = sequence[selected]
    return corrupted, targets, selected

def collate_masked(sequences, modalities, config, rng):
    corrupted, targets, masks = [], [], []
    max_length = max(map(len, sequences))
    assert max_length <= config.max_positions
    token_batch = sequences[0].new_full((len(sequences), max_length), PAD_ID)
    target_batch = sequences[0].new_full((len(sequences), max_length), -100)
    modality_batch = sequences[0].new_zeros((len(sequences), max_length))
    padding_mask = torch.ones((len(sequences), max_length), dtype=torch.bool, device=device)
    for i, (sequence, modality) in enumerate(zip(sequences, modalities)):
        masked, target, _ = corrupt_sequence(sequence, config, rng)
        n = len(sequence)
        token_batch[i, :n], target_batch[i, :n] = masked, target
        modality_batch[i, :n] = modality
        padding_mask[i, :n] = False
    return token_batch, modality_batch, padding_mask, target_batch

def interleave_sequences(speech_sequences, text_sequences, config, rng):
    mixed, modalities = [], []
    for speech, text in zip(speech_sequences, text_sequences):
        assert len(speech) == len(text)
        is_speech = torch.as_tensor(
            rng.random(len(speech)) < config.interleave_ratio, device=speech.device
        )
        mixed.append(torch.where(is_speech, speech, text))
        modalities.append(torch.where(is_speech, speech.new_zeros(()), speech.new_ones(())))
    return mixed, modalities

def mlm_branch(mlm_model, sequences, modalities, config, rng):
    tokens, modality_ids, padding, targets = collate_masked(sequences, modalities, config, rng)
    logits = mlm_model(tokens, modality_ids, padding)
    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)
    selected = targets != -100
    correct = (logits.argmax(-1)[selected] == targets[selected]).sum()
    return loss, int(correct), int(selected.sum()), (tokens, targets, logits)

def mlm_objective(mlm_model, indices, config, rng):
    speech = [speech_unit_sequences[int(i)] for i in indices]
    # Sample corpus text independently for the separate text-only objective.
    text_indices = rng.integers(0, len(unpaired_text_sequences), size=len(indices))
    text = [unpaired_text_sequences[int(i)] for i in text_indices]
    paired_predictions = [pusm_predicted_text_sequences[int(i)] for i in indices]
    speech_loss, speech_correct, speech_count, _ = mlm_branch(
        mlm_model, speech, [x.new_zeros(x.shape) for x in speech], config, rng
    )
    text_loss, text_correct, text_count, _ = mlm_branch(
        mlm_model, text, [x.new_ones(x.shape) for x in text], config, rng
    )
    mixed, mixed_modalities = interleave_sequences(speech, paired_predictions, config, rng)
    joint_loss, joint_correct, joint_count, joint_detail = mlm_branch(
        mlm_model, mixed, mixed_modalities, config, rng
    )
    total = (config.speech_weight * speech_loss + config.text_weight * text_loss
             + config.joint_weight * joint_loss)
    stats = {
        'speech_mlm_loss': float(speech_loss.detach()),
        'text_mlm_loss': float(text_loss.detach()),
        'joint_mlm_loss': float(joint_loss.detach()),
        'speech_mlm_correct': speech_correct, 'speech_mlm_tokens': speech_count,
        'text_mlm_correct': text_correct, 'text_mlm_tokens': text_count,
        'joint_mlm_correct': joint_correct, 'joint_mlm_tokens': joint_count,
    }
    return total, stats, joint_detail

torch.manual_seed(mlm_cfg.seed)
mlm_model = InterleavedMLM(len(mlm_symbols), mlm_cfg).to(device)
print(mlm_model)
print(f'MLM trainable parameters: {sum(p.numel() for p in mlm_model.parameters()):,}')

In [ ]:
def mlm_batches_for_epoch(n, batch_size, rng):
    size = n if batch_size is None else min(batch_size, n)
    order = rng.permutation(n)
    return [order[i:i + size] for i in range(0, n, size)]

@torch.no_grad()
def evaluate_mlm(mlm_model, config, seed):
    mlm_model.eval()
    rng = np.random.default_rng(seed)
    _, stats, detail = mlm_objective(mlm_model, np.arange(len(code_sequences)), config, rng)
    result = {k: v for k, v in stats.items() if not k.endswith('_correct') and not k.endswith('_tokens')}
    for branch in ['speech', 'text', 'joint']:
        result[f'{branch}_mlm_accuracy'] = (
            100.0 * stats[f'{branch}_mlm_correct'] / max(stats[f'{branch}_mlm_tokens'], 1)
        )
        result[f'{branch}_masked_tokens'] = stats[f'{branch}_mlm_tokens']
    result['joint_mlm_ppl'] = math.exp(min(result['joint_mlm_loss'], 20.0))
    return result, detail

def render_mlm_example(detail, row=0):
    inputs, targets, logits = detail
    valid = targets[row] != -100
    target_full = inputs[row].clone()
    target_full[valid] = targets[row][valid]
    restored = inputs[row].clone()
    restored[valid] = logits[row].argmax(-1)[valid]
    keep = target_full != PAD_ID
    def decode(sequence): return ' '.join(mlm_symbols[int(x)] for x in sequence[keep])
    return decode(target_full), decode(inputs[row]), decode(restored)

mlm_rng = np.random.default_rng(mlm_cfg.seed)
mlm_optimizer = torch.optim.AdamW(mlm_model.parameters(), lr=mlm_cfg.learning_rate)
mlm_optimizer.zero_grad(set_to_none=True)
mlm_history, mlm_updates, mlm_pending = [], 0, 0
best_joint_loss = math.inf

for epoch in range(1, mlm_cfg.epochs + 1):
    started = time.time()
    mlm_model.train()
    train_losses = []
    updates_at_start = mlm_updates
    epoch_batches = mlm_batches_for_epoch(len(code_sequences), mlm_cfg.batch_size, mlm_rng)
    for indices in epoch_batches:
        loss, _, _ = mlm_objective(mlm_model, indices, mlm_cfg, mlm_rng)
        (loss / mlm_cfg.update_frequency).backward()
        train_losses.append(float(loss.detach()))
        mlm_pending += 1
        if mlm_pending == mlm_cfg.update_frequency:
            mlm_optimizer.step()
            mlm_optimizer.zero_grad(set_to_none=True)
            mlm_pending = 0
            mlm_updates += 1

    if epoch == mlm_cfg.epochs and mlm_pending:
        correction = mlm_cfg.update_frequency / mlm_pending
        for parameter in mlm_model.parameters():
            if parameter.grad is not None:
                parameter.grad.mul_(correction)
        mlm_optimizer.step()
        mlm_optimizer.zero_grad(set_to_none=True)
        mlm_pending = 0
        mlm_updates += 1

    eval_stats, example_detail = evaluate_mlm(mlm_model, mlm_cfg, mlm_cfg.seed + 10000)
    row = {
        'epoch': epoch, 'num_updates': mlm_updates,
        'updates_this_epoch': mlm_updates - updates_at_start,
        'pending_microbatches': mlm_pending, 'microbatches': len(epoch_batches),
        'train_loss': float(np.mean(train_losses)), 'lr': mlm_optimizer.param_groups[0]['lr'],
        'seconds': time.time() - started, **eval_stats,
    }
    mlm_history.append(row)
    print(json.dumps(row, allow_nan=True))
    target_example, masked_example, hypothesis_example = render_mlm_example(example_detail)
    print(f'  joint TARGET: {target_example}')
    print(f'  joint MASKED: {masked_example}')
    print(f'  joint HYP:    {hypothesis_example}')
    if row['joint_mlm_loss'] < best_joint_loss:
        best_joint_loss = row['joint_mlm_loss']
        torch.save({'model': mlm_model.state_dict(), 'config': asdict(mlm_cfg),
                    'symbols': mlm_symbols, 'epoch': epoch, 'num_updates': mlm_updates,
                    'joint_mlm_loss': best_joint_loss}, WORK_DIR / 'checkpoint_mlm_best.pt')

torch.save({'model': mlm_model.state_dict(), 'config': asdict(mlm_cfg), 'symbols': mlm_symbols,
            'epoch': mlm_cfg.epochs, 'num_updates': mlm_updates}, WORK_DIR / 'checkpoint_mlm_last.pt')
mlm_history_df = pd.DataFrame(mlm_history)
mlm_history_df.to_csv(WORK_DIR / 'mlm_history.csv', index=False)
display(mlm_history_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
mlm_history_df.plot(
    x='epoch', y=['speech_mlm_loss', 'text_mlm_loss', 'joint_mlm_loss'],
    ax=axes[0], title='Stage-2 MLM losses',
)
mlm_history_df.plot(
    x='epoch', y=['speech_mlm_accuracy', 'text_mlm_accuracy', 'joint_mlm_accuracy'],
    ax=axes[1], title='Masked-token accuracy (%)',
)
plt.tight_layout()
plt.show()
print('All artifacts:', sorted(p.name for p in WORK_DIR.iterdir()))

## Stage 3: global PUSM after interleaved MLM

Inspired by `SpeechSyllableT5_transfer_learning_uniskip`, refine the trained shared encoder and text output head by matching their unmasked speech-output probabilities to corpus text unigram and skip-gram counts. Stage-1 predictions remain frozen speech inputs; Stage-2 weights are fine-tuned in place. Text contributes only aggregate statistics, with no random speech/text pairing or reference alignment.

Counts are accumulated over all batches before any L1 gradient is computed. A second pass backpropagates the exact global gradient with dropout disabled. Batch size controls memory; update frequency counts complete corpus passes. This compact stage uses PUSM unit inputs rather than the reference pipeline's acoustic prenet, and uses the final encoder layer. It optimizes only the global PUSM objective; the reference implementation also combines MLM losses on its gradient updates.

In [ ]:
@dataclass
class PostMLMPUSMConfig:
    epochs: int = int(os.environ.get('PUSM_POST_MLM_EPOCHS', '10'))
    batch_size: int | None = (None if os.environ.get('PUSM_POST_MLM_BATCH_SIZE', '5').lower() in {'none', 'full', 'all'} else int(os.environ.get('PUSM_POST_MLM_BATCH_SIZE', '5')))
    update_frequency: int = int(os.environ.get('PUSM_POST_MLM_UPDATE_FREQUENCY', '1'))
    learning_rate: float = 3e-4
    skipgram_size: int = cfg.skipgram_size
    max_text_positions: int = cfg.max_text_positions
    unigram_weight: float = 1.0
    skipgram_weight: float = 1.0
    seed: int = cfg.seed + 200

post_cfg = PostMLMPUSMConfig()
assert post_cfg.epochs >= 1 and post_cfg.update_frequency >= 1
assert post_cfg.batch_size is None or post_cfg.batch_size >= 1

class PostMLMPUSM(nn.Module):
    # Reuse the actual trained Stage-2 model; do not create a new projection.
    def __init__(self, trained_mlm):
        super().__init__()
        self.mlm = trained_mlm

    def forward(self, predicted_speech_units):
        tokens = predicted_speech_units.unsqueeze(0)
        modalities = torch.zeros_like(tokens)
        padding = torch.zeros_like(tokens, dtype=torch.bool)
        logits = self.mlm(tokens, modalities, padding)[0]
        # <mask> is an input corruption symbol, not a transcript unit.
        return logits[:, :len(symbols)].softmax(-1)

post_model = PostMLMPUSM(mlm_model).to(device)
post_rng = np.random.default_rng(post_cfg.seed)
post_optimizer = torch.optim.Adam(post_model.parameters(), lr=post_cfg.learning_rate,
                                  betas=cfg.adam_betas, eps=cfg.adam_eps)
post_optimizer.zero_grad(set_to_none=True)
post_history, post_updates, post_pending = [], 0, 0
post_best_uer = math.inf

@torch.no_grad()
def post_global_counts(batches):
    speech_counts, text_counts = None, None
    for speech_ids, text_ids in batches:
        speech = raw_counts([post_model(speech_unit_sequences[int(i)]) for i in speech_ids], len(symbols), post_cfg)
        text = raw_counts([one_hot_text(unpaired_text_sequences[int(i)], len(symbols)) for i in text_ids], len(symbols), post_cfg)
        if speech_counts is None:
            speech_counts = [torch.zeros_like(x) for x in speech]
            text_counts = [torch.zeros_like(x) for x in text]
        for dest, value in zip(speech_counts, speech): dest.add_(value)
        for dest, value in zip(text_counts, text): dest.add_(value)
    return speech_counts, text_counts

# Stage 3 uses corpus text as an aggregate distribution target, not utterance pairs.
assert len(unpaired_text_sequences) == len(speech_unit_sequences)
print(json.dumps(asdict(post_cfg), indent=2))
for epoch in range(1, post_cfg.epochs + 1):
    started = time.time()
    # eval() disables dropout but leaves autograd enabled in the gradient pass.
    post_model.eval()
    batches = batches_for_epoch(len(speech_unit_sequences), post_cfg.batch_size, post_rng)
    speech_counts, text_counts = post_global_counts(batches)
    differences = [speech - text for speech, text in zip(speech_counts, text_counts)]
    weights = [post_cfg.unigram_weight, post_cfg.skipgram_weight]
    signs = [difference.sign() * weight for difference, weight in zip(differences, weights)]
    uni_loss, skip_loss = [float(difference.abs().sum()) for difference in differences]
    updates_at_start = post_updates
    for speech_ids, _ in batches:
        counts = raw_counts([post_model(speech_unit_sequences[int(i)]) for i in speech_ids], len(symbols), post_cfg)
        surrogate = sum((count * sign).sum() for count, sign in zip(counts, signs))
        (surrogate / post_cfg.update_frequency).backward()
    post_pending += 1
    grad_norm = None
    if post_pending == post_cfg.update_frequency or epoch == post_cfg.epochs:
        if post_pending < post_cfg.update_frequency:
            for parameter in post_model.parameters():
                if parameter.grad is not None:
                    parameter.grad.mul_(post_cfg.update_frequency / post_pending)
        grad_norm = gradient_norm(post_model.parameters())
        post_optimizer.step()
        post_optimizer.zero_grad(set_to_none=True)
        post_updates += 1
        post_pending = 0
    metrics = evaluate(post_model, speech_unit_sequences, reference_sequences)
    row = {'epoch': epoch, 'num_updates': post_updates,
           'updates_this_epoch': post_updates - updates_at_start,
           'pending_global_passes': post_pending, 'microbatches': len(batches),
           'global_utterances': len(speech_unit_sequences),
           'loss': post_cfg.unigram_weight * uni_loss + post_cfg.skipgram_weight * skip_loss,
           'unigram_l1': uni_loss, 'skipgram_l1': skip_loss, 'grad_norm': grad_norm,
           'lr': post_cfg.learning_rate, 'seconds': time.time() - started,
           **{key: value for key, value in metrics.items() if key != 'hypotheses'}}
    post_history.append(row)
    print(json.dumps(row))
    print('  example REF:', ' '.join(symbols[x] for x in reference_sequences[0].tolist()))
    print('  example HYP:', ' '.join(symbols[x] for x in metrics['hypotheses'][0]))
    if metrics['uer'] < post_best_uer:
        post_best_uer = metrics['uer']
        torch.save({'model': mlm_model.state_dict(), 'config': asdict(post_cfg),
                    'mlm_config': asdict(mlm_cfg), 'symbols': mlm_symbols,
                    'epoch': epoch, 'num_updates': post_updates, 'uer': post_best_uer},
                   WORK_DIR / 'checkpoint_post_mlm_pusm_best.pt')

torch.save({'model': mlm_model.state_dict(), 'config': asdict(post_cfg),
            'mlm_config': asdict(mlm_cfg), 'symbols': mlm_symbols,
            'epoch': post_cfg.epochs, 'num_updates': post_updates},
           WORK_DIR / 'checkpoint_post_mlm_pusm_last.pt')
post_history_df = pd.DataFrame(post_history)
post_history_df.to_csv(WORK_DIR / 'post_mlm_pusm_history.csv', index=False)
display(post_history_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
post_history_df.plot(x='epoch', y=['loss', 'unigram_l1', 'skipgram_l1'], ax=axes[0], title='Post-MLM global PUSM')
post_history_df.plot(x='epoch', y='uer', ax=axes[1], title='Post-MLM UER (%)')
plt.tight_layout()
plt.show()

## What this reproduces (and what it intentionally omits)

Preserved: the 2,048-entry discrete speech input space, learned cluster-to-text projection, softmax unit probabilities, positional unigram plus lagged skip-gram L1 matching, independently sampled unpaired text, Adam hyperparameters, Fairseq-like special-token filtering for UER, best-UER checkpointing, and explicit optimizer-update accounting.

Omitted: Fairseq/Hydra registries, distributed training, the continuous-feature interleave branch, Slurm, indexed Fairseq text binaries, and the original even/odd Fairseq update-counter convention. The normalized mini-objective is intended to make a ten-utterance smoke test stable and readable. For a real experiment, scale back to the full manifests.